### Magnetism:PROPERTIES:



#### Determining if a magnetic solution is energetically favorable



We can force a total magnetic moment onto a unit cell and compute the total energy as function of the total magnetic moment. If there is a minimum in the energy, then we know there is a lower energy magnetic solution than a non-magnetic solution. We use incar:NUPDOWN to enforce the magnetic moment in the cell. Note that NUPDOWN can only be an integer. You cannot set it to be an arbitrary float.



In [1]:
from vasp import Vasp
from ase.lattice.cubic import BodyCenteredCubic

atoms = BodyCenteredCubic(directions=[[1, 0, 0],
                                      [0, 1, 0],
                                      [0, 0, 1]],
                                      size=(1, 1, 1),
                                      symbol='Fe')


calc = Vasp(label='bulk/Fe-bcc-fixedmagmom-{0:1.2f}'.format(0.0),
            xc='PBE',
            encut=300,
            kpts=[4, 4, 4],
            ispin=2,
            nupdown=0,
            atoms=atoms)

print(atoms.get_potential_energy())

-15.34226703

    -15.34226703



In [1]:
from vasp import Vasp
from ase.lattice.cubic import BodyCenteredCubic

atoms = BodyCenteredCubic(directions=[[1, 0, 0],
                                      [0, 1, 0],
                                      [0, 0, 1]],
                                      size=(1, 1, 1),
                                      symbol='Fe')

NUPDOWNS = [0.0, 2.0, 4.0, 5.0, 6.0, 8.0]
energies = []
for B in NUPDOWNS:
    calc = Vasp(label='bulk/Fe-bcc-fixedmagmom-{0:1.2f}'.format(B),
                xc='PBE',
                encut=300,
                kpts=[4, 4, 4],
                ispin=2,
                nupdown=B,
                atoms=atoms)
    energies.append(atoms.get_potential_energy())

if None in energies:
    calc.abort()

import matplotlib.pyplot as plt
plt.plot(NUPDOWNS, energies)
plt.xlabel('Total Magnetic Moment')
plt.ylabel('Energy (eV)')
plt.savefig('images/Fe-fixedmagmom.png')

![img](./images/Fe-fixedmagmom.png "Total energy vs. total magnetic moment for bcc Fe.")

You can see here there is a minimum in energy at a total magnetic moment somewhere between 4 and 5. There are two Fe atoms in the unit cell, which means the magnetic moment on each atom must be about 2.5 Bohr-magnetons. This is a good guess for a real calculation. Note that VASP [recommends](http://cms.mpi.univie.ac.at/vasp/guide/node100.html#SECTION00099000000000000000) you overestimate the magnetic moment guesses if you are looking for ferromagnetic solutions.

To run a spin-polarized calculation with initial guesses on each atom, we must set the magnetic moment on the atoms. Here we set it through the `magmom` attribute on the atom. In the example after this, we set it in the `Atoms` object.



In [ ]:
from vasp import Vasp
from ase.lattice.cubic import BodyCenteredCubic

atoms = BodyCenteredCubic(directions=[[1, 0, 0],
                                      [0, 1, 0],
                                      [0, 0, 1]],
                                      size=(1, 1, 1),
                                      symbol='Fe')

# set magnetic moments on each atom
for atom in atoms:
    atom.magmom = 2.5

calc = Vasp(label='bulk/Fe-bcc-sp-1',
            xc='PBE',
            encut=300,
            kpts=[4, 4, 4],
            ispin=2,
            lorbit=11, # you need this for individual magnetic moments
            atoms=atoms)

e = atoms.get_potential_energy()
B = atoms.get_magnetic_moment()
magmoms = atoms.get_magnetic_moments()

print('Total magnetic moment is {0:1.2f} Bohr-magnetons'.format(B))
print('Individual moments are {0} Bohr-magnetons'.format(magmoms))

    Total magnetic moment is -0.01 Bohr-magnetons
    Individual moments are [-0.013 -0.013] Bohr-magnetons



#### Antiferromagnetic spin states



In an antiferromagnetic material, there are equal numbers of spin up and down electrons that align in a regular pattern, but pointing in opposite directions so that there is no net magnetism. It is possible to model this by setting the magnetic moments on each mod:ase.Atom object. incar:lreal



In [ ]:
from vasp import Vasp
from ase import Atom, Atoms

atoms = Atoms([Atom('Fe', [0.00,  0.00,  0.00], magmom=5),
               Atom('Fe', [4.3,   4.3,   4.3],  magmom=-5),
               Atom('O', [2.15,  2.15,  2.15], magmom=0),
               Atom('O', [6.45,  6.45,  6.45], magmom=0)],
               cell=[[4.3,    2.15,    2.15],
                     [2.15,    4.3,     2.15],
                     [2.15,    2.15,    4.3]])

ca = Vasp(label='bulk/afm-feo',
          encut=350,
          prec='Normal',
          ispin=2,
          nupdown=0, # this forces a non-magnetic solution
          lorbit=11, # to get individual moments
          lreal=False,
          atoms=atoms)
print('Magnetic moments = ', atoms.get_magnetic_moments())
print('Total magnetic moment = ', atoms.get_magnetic_moment())

    Magnetic moments =  [-0.061 -0.061  0.063  0.063]
    Total magnetic moment =  -5e-06

You can see that even though the total magnetic moment is 0, there is a spin on both Fe atoms, and they are pointing in opposite directions. Hence, the sum of spins is zero, and this arrangement is called anti-ferromagnetic.



#### NiO-FeO formation energies with magnetism

